# Critical damping and resonance

Run this lab in a kernel with the repository installed. A unit force step
takes an SDOF system from rest to $x=1/k$. The damping ratio is
$\zeta=c/(2\sqrt{km})$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from modal_lab import absorber_system, critical_step, damped_oscillator, linear_frf, simulate
from modal_lab.plotting import plot_damping, plot_frf

## 1. Compare damping regimes

At $\zeta=1$, $x(t)=[1-(1+\omega_nt)e^{-\omega_nt}]/k$. Compare this
exact curve with the numerical result. Look for overshoot when the
system is underdamped and a slow approach when it is overdamped.

In [ ]:
time = np.linspace(0, 22, 701)
presets = ("underdamped", "critical", "overdamped")
responses = {label: simulate(damped_oscillator(label), time, force=lambda _: 1.0)
             for label in presets}
exact = critical_step(time)
print(f"Critical max error: {np.max(abs(responses['critical'].displacement[:, 0] - exact)):.2e} m")
plot_damping(responses, exact=(time, exact))
plt.show()

## 2. Sweep the drive frequency

The complex displacement is $F/(k-m\omega^2+i\omega c)$.
Below $\zeta=1/\sqrt{2}$, the peak occurs at
$\omega_n\sqrt{1-2\zeta^2}$. Compare the theoretical and sampled peaks.

In [ ]:
omega = np.linspace(0, 2, 1001)
curves = {}
for zeta in (0.05, 0.15, 0.35):
    amplitude = linear_frf(damped_oscillator(zeta=zeta), omega)[:, 0]
    curves[f"ζ = {zeta}"] = omega, amplitude
    print(f"ζ={zeta}: peak {omega[np.argmax(abs(amplitude))]:.4f}, "
          f"theory {np.sqrt(1 - 2*zeta**2):.4f} rad/s")
plot_frf(curves, ylabel="Receptance (m/N)")
plt.show()

## 3. Protect a driven primary structure

A tuned linear attachment splits the resonant region and reduces the
response near the uncontrolled peak. Vary the tuning frequency and the
attachment damping. Check the full sweep to see whether the attachment
also introduces amplification away from its tuning frequency.

In [ ]:
omega = np.linspace(0.65, 1.35, 500)
primary = damped_oscillator(zeta=0.03)
tmd = absorber_system(tune=1.0, attachment_damping=0.025)
bare = linear_frf(primary, omega)[:, 0]
controlled = linear_frf(tmd, omega, load=[1, 0])[:, 0]
print(f"Peak amplitudes: bare {abs(bare).max():.3f}, TMD {abs(controlled).max():.3f} m/N")
plot_frf({"Uncontrolled": (omega, bare), "Linear TMD": (omega, controlled)},
         ylabel="Receptance (m/N)")
plt.show()